In [ ]:
# =============================================================================
# Transformer版本说明（中文强制）
# 【版本身份】
# - 版本：V40A
# - 实验ID：FREQ2_MULTI_FREQ_POS
# - 创建日期：2026-08-05
# - 当前状态：本地实现完成，平台总分、四项指标、耗时和覆盖率待平台验证
# 【直接父级】
# - 父级版本：V33C
# - 父级Notebook：realtime/A/Transformer_realtime_v33C.ipynb
# 【研究目标】
# - 用更丰富的固定位置先验增强时序归纳偏置。
# 【核心算法流程】
# 1. 查询历史中证1000成分股和官方一分钟行情。
# 2. 构造240根窗口与次日收益标签，继承V33C预处理与软多空。
# 3. 按版本加入两段辅助头、多频位置、并行TCN、FFT池化、波动率标签或日程重测。
# 4. 使用AdamW与六轮日程训练并输出每日分数。
# 5. 输出date、instrument、score三列。
# 【相对父级的演变优化】
# - 多频位置编码：快速正弦(base=100)+线性斜坡，零门控
# 【继承且保持不变】
# - 父级位置×软多空、数据、损失主干、优化器与推理保持不变。
# - 绿色：官方字段、模型内部结构或训练目标。
# 【关键配置】
# - V39D辅助权重0.02；V40A/B/C零门控初始等价V33C；V40D标签缩放；V40E批量4096；V40F Patch=4。
# - 训练区间2019-01-01至2024-12-31。
# 【合规边界】
# - 无未来信息、无外部权重、无CUDA扩展。
# 【验证与证据状态】
# - 行为合同覆盖结构与CPU短训练；平台证据待回填。
# 【文件关系】
# - 当前Notebook：realtime/A/Transformer_realtime_v40A.ipynb。
# - 生成器：scripts/build_transformer_realtime_v39d_v40_batch.py。
# - 主路线图：docs/Transformer版本升级优化分支路线图.md。
# =============================================================================

import gc
import logging
import math
from collections.abc import Mapping
import os
import random
import time

import numpy as np
import pandas as pd

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Sampler, TensorDataset


VERSION_NAME = "V40A"
PARENT_VERSION = "V33C"
EXPERIMENT_ID = "FREQ2_MULTI_FREQ_POS"

TRAIN_START = "2019-01-01"
TRAIN_END = "2024-12-31 23:59:59"
TRAIN_TABLE = "bigalpha_2026_stock_bar1m"
POOL_TABLE = "bigalpha_2026_instruments"
SEED = 20260717
SEQ_LEN = 240
PATCH_SIZE = 2
TOKEN_SEQ_LEN = SEQ_LEN // PATCH_SIZE
QUERY_INSTRUMENT_CHUNK = 40
FINITE_CHECK_BLOCK_ITEMS = 1_000_000
BATCH_SIZE = 2048
INFER_BATCH_SIZE = 2048
INFER_BUFFER_DAYS = 20
EPOCHS = 6
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.10
GRAD_CLIP_NORM = 1.0
MIN_CROSS_SECTION = 128
LOSS_IC_WEIGHT = 0.50
LOSS_PAIR_WEIGHT = 0.30
LOSS_REG_WEIGHT = 0.20
PAIR_OFFSETS = (1, 7, 31, 127)
SOFT_LS_WEIGHT = 0.02
SOFT_LS_TARGET_FRACTION = 0.10
SOFT_LS_MIN_TEMPERATURE = 0.05
SOFT_LS_MAX_TEMPERATURE = 5.00
SOFT_LS_BISECTION_STEPS = 12
SOFT_LS_EPS = 1e-6
RAW_FEATURE_COLS = (
    "open",
    "high",
    "low",
    "close",
    "bid_price1",
    "ask_price1",
    "volume",
    "amount",
    "bid_volume1",
    "ask_volume1",
)
FEATURE_COLS = RAW_FEATURE_COLS
LOG1P_COLS = ("volume", "amount", "bid_volume1", "ask_volume1")
MODEL_CONFIG = {
    "n_features": len(FEATURE_COLS),
    "d_model": 128,
    "nhead": 4,
    "nlayers": 3,
    "dim_ff": 320,
    "dropout": 0.1,
    "seq_len": SEQ_LEN,
    "patch_size": PATCH_SIZE,
}
LOCAL_SMOKE_WARNING = (
    "2024 is inside the V40A training period; local evaluation is an in-sample "
    "pipeline smoke test, not an out-of-sample score."
)
MIN_TRAINABLE_PARAMETERS = 100_000
MAX_TRAINABLE_PARAMETERS = 100_000_000
LOGGER = logging.getLogger("transformer_realtime_v40a")
LOGGER.setLevel(logging.INFO)
LOGGER.propagate = False
if not any(
    getattr(handler, "_transformer_realtime_v40a", False)
    for handler in LOGGER.handlers
):
    _log_handler = logging.StreamHandler()
    _log_handler.setFormatter(
        logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    )
    _log_handler._transformer_realtime_v40a = True
    LOGGER.addHandler(_log_handler)


def seed_everything(seed):
    """Seed Python, NumPy, and PyTorch for reproducible training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class V2EnhancedTransformer(nn.Module):
    """Post-LN V6 backbone over model-internal fixed-size temporal patches."""

    def __init__(
        self,
        n_features=len(FEATURE_COLS),
        d_model=128,
        nhead=4,
        nlayers=3,
        dim_ff=320,
        dropout=0.1,
        seq_len=SEQ_LEN,
        patch_size=PATCH_SIZE,
    ):
        super().__init__()
        if seq_len % patch_size != 0:
            raise ValueError("seq_len must be divisible by patch_size")
        self.raw_seq_len = int(seq_len)
        self.patch_size = int(patch_size)
        self.token_seq_len = self.raw_seq_len // self.patch_size
        self.n_features = int(n_features)
        self.input_projection = nn.Linear(n_features * patch_size, d_model)
        position = torch.zeros(1, self.token_seq_len, d_model)
        position_index = torch.arange(
            self.token_seq_len, dtype=torch.float32
        ).reshape(1, -1, 1)
        frequency = torch.pow(
            10000.0,
            -torch.arange(0, d_model, 2, dtype=torch.float32) / d_model,
        )
        position[..., 0::2] = torch.sin(position_index * frequency)
        position[..., 1::2] = torch.cos(position_index * frequency)
        self.register_buffer("position", position, persistent=False)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(layer, nlayers, enable_nested_tensor=False)
        self.temporal_score = nn.Linear(d_model, 1)
        self.fusion = nn.Sequential(
            nn.LayerNorm(d_model * 3), nn.Linear(d_model * 3, d_model),
            nn.GELU(), nn.Dropout(dropout),
        )
        position_fast = torch.zeros(1, self.token_seq_len, d_model)
        frequency_fast = torch.pow(
            100.0,
            -torch.arange(0, d_model, 2, dtype=torch.float32) / d_model,
        )
        position_fast[..., 0::2] = torch.sin(position_index * frequency_fast)
        position_fast[..., 1::2] = torch.cos(position_index * frequency_fast)
        self.register_buffer("position_fast", position_fast, persistent=False)
        position_ramp = torch.linspace(
            0.0, 1.0, self.token_seq_len, dtype=torch.float32
        ).reshape(1, -1, 1)
        self.register_buffer("position_ramp", position_ramp, persistent=False)
        self.position_extra_gate = nn.Parameter(torch.zeros(1))

        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model // 2, 1),
        )


    def forward(self, x):
        if x.ndim != 3 or x.shape[1:] != (self.raw_seq_len, self.n_features):
            raise ValueError(
                f"model input must have shape (N, {self.raw_seq_len}, {self.n_features})"
            )
        patches = x.reshape(
            x.shape[0], self.token_seq_len, self.patch_size * self.n_features
        )
        position_extra = self.position_fast + self.position_ramp
        encoded = self.encoder(
            self.input_projection(patches)
            + self.position
            + self.position_extra_gate * position_extra
        )
        global_pool = encoded.mean(dim=1)
        final_pool = encoded[:, -1]
        attention = torch.softmax(self.temporal_score(encoded).squeeze(-1), dim=1)
        attention_pool = torch.sum(encoded * attention.unsqueeze(-1), dim=1)
        fused = self.fusion(torch.cat((global_pool, final_pool, attention_pool), dim=-1))
        return self.head(fused).squeeze(-1)


def count_trainable_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def validate_parameter_count(model):
    parameter_count = count_trainable_parameters(model)
    if not MIN_TRAINABLE_PARAMETERS <= parameter_count <= MAX_TRAINABLE_PARAMETERS:
        raise ValueError(
            "trainable parameter count must be within "
            f"[{MIN_TRAINABLE_PARAMETERS}, {MAX_TRAINABLE_PARAMETERS}], got {parameter_count}"
        )
    return parameter_count


def prepare_raw_feature_frame(raw, assume_sorted=False):
    """Apply only independent, rule-compliant transformations to V2 raw fields."""
    required = {"date", "instrument", *FEATURE_COLS}
    missing = sorted(required.difference(raw.columns))
    if missing:
        raise ValueError(f"raw data is missing required columns: {missing}")

    prepared = raw.loc[:, ["date", "instrument", *FEATURE_COLS]].copy()
    if not pd.api.types.is_datetime64_any_dtype(prepared["date"]):
        prepared["date"] = pd.to_datetime(prepared["date"], errors="coerce")
    if prepared["date"].isna().any():
        raise ValueError("raw data contains null or invalid timestamps")
    prepared["instrument"] = prepared["instrument"].astype(str, copy=False)
    for column in FEATURE_COLS:
        if not pd.api.types.is_numeric_dtype(prepared[column]):
            prepared[column] = pd.to_numeric(prepared[column], errors="coerce")
    for column in LOG1P_COLS:
        prepared[column] = np.log1p(prepared[column].clip(lower=0))

    if assume_sorted:
        return prepared.reset_index(drop=True)
    return prepared.sort_values(["instrument", "date"], kind="mergesort").reset_index(drop=True)


def build_raw_sequences(
    raw,
    start_date,
    end_date,
    mode,
    seq_len,
    assume_sorted=False,
):
    """Build close-ended minute windows and optional next-day return labels."""
    if not isinstance(mode, str) or mode not in {"train", "infer"}:
        raise ValueError(f"mode must be exactly 'train' or 'infer', got {mode!r}")
    if isinstance(seq_len, bool) or not isinstance(seq_len, (int, np.integer)) or seq_len <= 0:
        raise ValueError("seq_len must be a positive integer")

    prepared = prepare_raw_feature_frame(raw, assume_sorted=assume_sorted)
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    start64 = start.to_datetime64()
    end64 = end.to_datetime64()
    x_parts = []
    y_parts = []
    date_parts = []
    instrument_parts = []

    for instrument, instrument_frame in prepared.groupby("instrument", sort=False):
        timestamps = instrument_frame["date"].to_numpy(dtype="datetime64[ns]")
        if len(timestamps) < seq_len:
            continue
        sample_days = timestamps.astype("datetime64[D]")
        day_end = np.empty(len(sample_days), dtype=bool)
        day_end[:-1] = sample_days[:-1] != sample_days[1:]
        day_end[-1] = True
        close_positions = np.flatnonzero(day_end)
        close_timestamps = timestamps[close_positions]
        close_days = sample_days[close_positions]
        close_values = instrument_frame["close"].to_numpy(dtype=np.float64)[close_positions]

        valid = (
            (close_timestamps >= start64)
            & (close_timestamps <= end64)
            & (close_positions + 1 >= seq_len)
        )
        targets = None
        if mode == "train":
            targets = np.full(len(close_positions), np.nan, dtype=np.float64)
            target_timestamps = np.full(
                len(close_positions),
                np.datetime64("NaT"),
                dtype="datetime64[ns]",
            )
            if len(close_positions) > 1:
                current_close = close_values[:-1]
                next_close = close_values[1:]
                return_valid = (
                    np.isfinite(current_close)
                    & np.isfinite(next_close)
                    & (current_close != 0)
                )
                valid_indices = np.flatnonzero(return_valid)
                targets[valid_indices] = (
                    next_close[valid_indices] / current_close[valid_indices]
                ) - 1.0
                target_timestamps[:-1] = close_timestamps[1:]
            valid &= np.isfinite(targets) & (target_timestamps <= end64)

        selected_indices = np.flatnonzero(valid)
        if selected_indices.size == 0:
            continue
        selected_close_positions = close_positions[selected_indices]
        feature_values = instrument_frame.loc[:, FEATURE_COLS].to_numpy(dtype=np.float32)
        window_view = np.lib.stride_tricks.sliding_window_view(
            feature_values,
            window_shape=seq_len,
            axis=0,
        )
        window_starts = selected_close_positions - seq_len + 1
        selected_windows = np.swapaxes(window_view[window_starts], 1, 2)
        x_parts.append(selected_windows)
        date_parts.append(close_days[selected_indices])
        instrument_parts.append(
            np.full(selected_indices.size, instrument, dtype=object)
        )
        if mode == "train":
            y_parts.append(targets[selected_indices])

    if not x_parts:
        raise RuntimeError(f"no usable sample exists for mode={mode!r} date range {start_date} to {end_date}")

    x_all = np.concatenate(x_parts, axis=0).astype(np.float32, copy=False)
    dates_all = np.concatenate(date_parts)
    instruments_all = np.concatenate(instrument_parts)
    order = np.lexsort((instruments_all, dates_all.astype(np.int64)))
    x = np.ascontiguousarray(x_all[order])
    keys = pd.DataFrame(
        {
            "date": pd.to_datetime(dates_all[order].astype("datetime64[ns]")),
            "instrument": instruments_all[order],
        },
        columns=("date", "instrument"),
    )
    if keys.duplicated(["date", "instrument"]).any():
        raise RuntimeError(f"duplicate keys produced for mode={mode!r} date range {start_date} to {end_date}")

    if mode == "train":
        y_all = np.concatenate(y_parts).astype(np.float32, copy=False)
        y = np.ascontiguousarray(y_all[order])
    else:
        y = None
    return x, y, keys


def _finite_values_in_blocks(values, block_items=FINITE_CHECK_BLOCK_ITEMS):
    flat = np.asarray(values).reshape(-1)
    finite_parts = []
    for start in range(0, len(flat), block_items):
        block = flat[start : start + block_items]
        finite_parts.append(block[np.isfinite(block)])
    return np.concatenate(finite_parts) if finite_parts else flat[:0].copy()


def fit_preprocess_inplace(x, y, block_rows=250_000):
    """Fit independent field statistics in contiguous blocks and transform in place."""
    x_array = np.asarray(x)
    y_array = np.asarray(y)
    if x_array.dtype != np.float32 or not x_array.flags.c_contiguous:
        raise ValueError("training tensor must be contiguous float32 storage")
    if y_array.dtype != np.float32 or not y_array.flags.c_contiguous:
        raise ValueError("training labels must be contiguous float32 storage")
    if not x_array.flags.writeable or not y_array.flags.writeable:
        raise ValueError("training storage must be writable")
    if x_array.ndim != 3 or x_array.shape[2] != len(FEATURE_COLS):
        raise ValueError("training tensor must have shape (N, seq_len, feature width 10)")
    if y_array.ndim != 1 or len(y_array) != len(x_array) or len(y_array) == 0:
        raise ValueError("training labels must be a non-empty 1-D array matching samples")
    if block_rows <= 0:
        raise ValueError("block_rows must be positive")

    flat = x_array.reshape(-1, x_array.shape[2])
    feature_count = len(FEATURE_COLS)
    finite_sum = np.zeros(feature_count, dtype=np.float64)
    finite_count = np.zeros(feature_count, dtype=np.int64)
    for start in range(0, len(flat), block_rows):
        block = flat[start : start + block_rows]
        finite = np.isfinite(block)
        finite_sum += np.where(finite, block, 0.0).sum(axis=0, dtype=np.float64)
        finite_count += finite.sum(axis=0, dtype=np.int64)

    fill64 = np.divide(
        finite_sum,
        finite_count,
        out=np.zeros(feature_count, dtype=np.float64),
        where=finite_count > 0,
    )
    fill = fill64.astype(np.float32)
    mean = fill.copy()

    squared_total = np.zeros(feature_count, dtype=np.float64)
    mean64 = mean.astype(np.float64)
    for start in range(0, len(flat), block_rows):
        block = flat[start : start + block_rows]
        bounded = np.where(np.isfinite(block), block, fill).astype(np.float64)
        bounded -= mean64
        np.square(bounded, out=bounded)
        squared_total += bounded.sum(axis=0, dtype=np.float64)
    std = np.sqrt(squared_total / len(flat)).astype(np.float32)
    std = np.where(np.isfinite(std) & (std >= 1e-6), std, 1.0).astype(np.float32)

    for start in range(0, len(flat), block_rows):
        block = flat[start : start + block_rows]
        np.copyto(block, fill, where=~np.isfinite(block))
        block -= mean
        block /= std

    finite_y = _finite_values_in_blocks(y_array, block_items=block_rows)
    if finite_y.size == 0:
        raise ValueError("training labels must contain at least one finite value")
    label_clip = np.percentile(finite_y, [1, 99]).astype(np.float32)
    del finite_y
    np.clip(y_array, label_clip[0], label_clip[1], out=y_array)
    _assert_finite_array("training labels", y_array)
    return {
        "fill": np.ascontiguousarray(fill),
        "mean": np.ascontiguousarray(mean),
        "std": np.ascontiguousarray(std),
        "label_clip": np.ascontiguousarray(label_clip),
    }


def _preprocess_vector(preprocess, name):
    values = np.asarray(preprocess[name], dtype=np.float32)
    if values.shape != (len(FEATURE_COLS),):
        raise ValueError(f"preprocess {name!r} must match feature width {len(FEATURE_COLS)}")
    return values


def apply_preprocess(x, preprocess):
    """Apply saved imputation and z-score parameters without refitting."""
    x_array = np.asarray(x, dtype=np.float32)
    if x_array.ndim != 3:
        raise ValueError("input tensor must have shape (N, seq_len, feature width 10)")
    if x_array.shape[2] != len(FEATURE_COLS):
        raise ValueError(f"input tensor feature width must be {len(FEATURE_COLS)}")

    fill = _preprocess_vector(preprocess, "fill")
    mean = _preprocess_vector(preprocess, "mean")
    std = _preprocess_vector(preprocess, "std")
    imputed = np.where(np.isfinite(x_array), x_array, fill)
    transformed = (imputed - mean) / std
    return np.ascontiguousarray(transformed.astype(np.float32, copy=False))


def apply_preprocess_inplace(x, preprocess, block_rows=250_000):
    """Apply saved field statistics to writable contiguous float32 storage."""
    x_array = np.asarray(x)
    if x_array.dtype != np.float32 or not x_array.flags.c_contiguous:
        raise ValueError("input tensor must be contiguous float32 storage")
    if not x_array.flags.writeable:
        raise ValueError("input tensor must be writable")
    if x_array.ndim != 3 or x_array.shape[2] != len(FEATURE_COLS):
        raise ValueError(
            f"input tensor must have shape (N, seq_len, feature width {len(FEATURE_COLS)})"
        )
    if block_rows <= 0:
        raise ValueError("block_rows must be positive")

    fill = _preprocess_vector(preprocess, "fill")
    mean = _preprocess_vector(preprocess, "mean")
    std = _preprocess_vector(preprocess, "std")
    flat = x_array.reshape(-1, x_array.shape[2])
    for start in range(0, len(flat), block_rows):
        block = flat[start : start + block_rows]
        np.copyto(block, fill, where=~np.isfinite(block))
        block -= mean
        block /= std
    return x_array


def _default_query(sql, filters=None, compression=False):
    import dai

    return dai.query(sql, filters=filters, compression=compression)


def pick_bar1m_table(datasources):
    if not isinstance(datasources, Mapping):
        raise TypeError("datasources must be a mapping containing a 1-minute datasource")
    for key in ("bar1m", "bar_1m", "e2e_bar1m"):
        if key in datasources:
            table = datasources[key]
            if not isinstance(table, str) or not table.strip():
                raise ValueError("1-minute datasource must be a non-empty table name")
            return table
    raise KeyError("a 1-minute datasource is required")


def _query_result_to_frame(result):
    if isinstance(result, pd.DataFrame):
        return result
    for method_name in ("df", "to_dataframe", "to_pandas"):
        method = getattr(result, method_name, None)
        if callable(method):
            frame = method()
            if not isinstance(frame, pd.DataFrame):
                raise TypeError(f"query result {method_name}() did not return a DataFrame")
            return frame
    raise TypeError("query result must be a DataFrame or expose df(), to_dataframe(), or to_pandas()")


def normalise_training_pool(pool):
    required = {"date", "instrument"}
    missing = sorted(required.difference(pool.columns))
    if missing:
        raise ValueError(f"training pool is missing required columns: {missing}")

    normalised = pool.loc[:, ["date", "instrument"]].copy()
    normalised["date"] = pd.to_datetime(
        normalised["date"], errors="coerce", format="mixed"
    )
    if normalised["date"].isna().any():
        raise ValueError("training pool contains null or invalid dates")
    normalised["date"] = normalised["date"].dt.normalize()
    if normalised["instrument"].isna().any():
        raise ValueError("training pool contains null instruments")
    normalised["instrument"] = normalised["instrument"].astype(str).str.strip()
    if normalised["instrument"].eq("").any():
        raise ValueError("training pool contains empty instruments")
    if normalised.empty:
        raise ValueError("training pool is empty")
    if normalised.duplicated(["date", "instrument"]).any():
        raise ValueError("training pool contains duplicate date/instrument keys")
    return normalised.sort_values(
        ["date", "instrument"], kind="mergesort"
    ).reset_index(drop=True)


def query_training_pool(query_fn=None):
    started = time.perf_counter()
    query = query_fn or _default_query
    sql = (
        "select date, instrument from bigalpha_2026_instruments "
        "order by date, instrument"
    )
    frame = _query_result_to_frame(
        query(
            sql,
            filters={"date": [TRAIN_START, TRAIN_END]},
            compression=False,
        )
    )
    normalised = normalise_training_pool(frame)
    daily_counts = normalised.groupby("date", sort=True).size()
    LOGGER.info(
        "training pool query complete: rows=%s days=%s instruments=%s "
        "daily_min=%s daily_median=%.1f daily_max=%s elapsed_seconds=%.2f",
        len(normalised),
        normalised["date"].nunique(),
        normalised["instrument"].nunique(),
        int(daily_counts.min()),
        float(daily_counts.median()),
        int(daily_counts.max()),
        time.perf_counter() - started,
    )
    return normalised


def filter_training_samples(
    x,
    y,
    keys,
    eligible_pool=None,
    eligible_index=None,
):
    x_array = np.asarray(x, dtype=np.float32)
    y_array = np.asarray(y, dtype=np.float32)
    if x_array.ndim != 3 or x_array.shape[2] != len(FEATURE_COLS):
        raise ValueError("candidate tensor must have shape (N, seq_len, feature width 10)")
    if y_array.ndim != 1 or len(y_array) != len(x_array):
        raise ValueError("candidate labels must match candidate samples")
    required = {"date", "instrument"}
    if required.difference(keys.columns):
        raise ValueError("candidate keys must contain date and instrument")

    candidate_keys = keys.loc[:, ["date", "instrument"]].copy()
    candidate_keys["date"] = pd.to_datetime(
        candidate_keys["date"], errors="coerce", format="mixed"
    )
    if candidate_keys["date"].isna().any():
        raise ValueError("candidate keys contain invalid dates")
    candidate_keys["date"] = candidate_keys["date"].dt.normalize()
    if candidate_keys["instrument"].isna().any():
        raise ValueError("candidate keys contain null instruments")
    candidate_keys["instrument"] = candidate_keys["instrument"].astype(str).str.strip()
    if candidate_keys["instrument"].eq("").any():
        raise ValueError("candidate keys contain empty instruments")
    if len(candidate_keys) != len(x_array):
        raise ValueError("candidate keys must match candidate samples")
    if candidate_keys.duplicated(["date", "instrument"]).any():
        raise ValueError("candidate keys contain duplicate date/instrument keys")

    candidate_index = pd.MultiIndex.from_frame(
        candidate_keys[["date", "instrument"]]
    )
    if eligible_index is None:
        if eligible_pool is None:
            raise ValueError("eligible_pool or eligible_index is required")
        pool = normalise_training_pool(eligible_pool)
        eligible_index = pd.MultiIndex.from_frame(pool[["date", "instrument"]])
    elif not isinstance(eligible_index, pd.MultiIndex) or eligible_index.nlevels != 2:
        raise TypeError("eligible_index must be a two-level pandas MultiIndex")
    keep = candidate_index.isin(eligible_index)
    rejected = int((~keep).sum())
    return (
        np.ascontiguousarray(x_array[keep]),
        np.ascontiguousarray(y_array[keep]),
        candidate_keys.loc[keep].reset_index(drop=True),
        rejected,
    )


def write_training_samples(
    storage_x,
    storage_y,
    storage_dates,
    offset,
    x_part,
    y_part,
    keys,
):
    if isinstance(offset, bool) or not isinstance(offset, (int, np.integer)) or offset < 0:
        raise ValueError("training write offset must be a non-negative integer")
    x_array = np.asarray(x_part, dtype=np.float32)
    y_array = np.asarray(y_part, dtype=np.float32)
    if x_array.ndim != 3 or x_array.shape[1:] != storage_x.shape[1:]:
        raise ValueError("training chunk shape does not match preallocated storage")
    if y_array.ndim != 1 or len(y_array) != len(x_array):
        raise ValueError("training chunk labels must match samples")
    if len(keys) != len(x_array):
        raise ValueError("training chunk keys must match samples")
    dates = pd.to_datetime(keys["date"], errors="coerce").dt.normalize()
    if dates.isna().any():
        raise ValueError("training chunk keys contain invalid dates")
    date_ids = dates.to_numpy(dtype="datetime64[D]").astype(np.int64)
    end = offset + len(x_array)
    if end > len(storage_x) or end > len(storage_y) or end > len(storage_dates):
        raise RuntimeError(
            f"training sample capacity exceeded: end={end} capacity={len(storage_x)}"
        )
    storage_x[offset:end] = x_array
    storage_y[offset:end] = y_array
    storage_dates[offset:end] = date_ids
    return end


def query_raw_bars(table, instruments, query_fn=None):
    started = time.perf_counter()
    instrument_list = [str(instrument) for instrument in instruments]
    if len(instrument_list) > QUERY_INSTRUMENT_CHUNK:
        raise ValueError(
            f"bar query cannot request more than {QUERY_INSTRUMENT_CHUNK} instruments, got {len(instrument_list)}"
        )
    columns = ("date", "instrument", *FEATURE_COLS)
    sql = f"select {', '.join(columns)} from {table} order by instrument, date"
    query = query_fn or _default_query
    LOGGER.info(
        "training raw-bar query: table=%s date_start=%s date_end=%s chunk_size=%s",
        table,
        TRAIN_START,
        TRAIN_END,
        len(instrument_list),
    )
    frame = _query_result_to_frame(
        query(
            sql,
            filters={
                "date": [TRAIN_START, TRAIN_END],
                "instrument": instrument_list,
            },
            compression=True,
        )
    )
    LOGGER.info(
        "training raw-bar query complete: table=%s instruments=%s rows=%s elapsed_seconds=%.2f",
        table,
        len(instrument_list),
        len(frame),
        time.perf_counter() - started,
    )
    return frame


def iter_chunks(values, chunk_size):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    for start in range(0, len(values), chunk_size):
        yield values[start : start + chunk_size]


def _assert_finite_array(name, values, *, block_items=FINITE_CHECK_BLOCK_ITEMS):
    if (
        isinstance(block_items, bool)
        or not isinstance(block_items, (int, np.integer))
        or block_items <= 0
    ):
        raise ValueError("finite-check block_items must be a positive integer")
    flat = np.asarray(values).reshape(-1)
    for start in range(0, len(flat), block_items):
        if not np.isfinite(flat[start : start + block_items]).all():
            raise ValueError(f"non-finite {name} values are not allowed")


class DateBatchSampler(Sampler):
    """Yield one complete trading-date cross-section per optimization step."""

    def __init__(
        self,
        date_ids,
        *,
        shuffle_dates=True,
        seed=SEED,
        min_samples=MIN_CROSS_SECTION,
        max_samples=BATCH_SIZE,
    ):
        values = np.asarray(date_ids)
        if values.ndim != 1 or values.size == 0:
            raise ValueError("date_ids must be a non-empty one-dimensional array")
        if not np.issubdtype(values.dtype, np.integer):
            raise TypeError("date_ids must use an integer dtype")
        if min_samples <= 1:
            raise ValueError("min_samples must be greater than one")
        if max_samples < min_samples:
            raise ValueError("max_samples must be at least min_samples")

        order = np.argsort(values, kind="stable")
        sorted_dates = values[order]
        boundaries = np.flatnonzero(np.diff(sorted_dates)) + 1
        starts = np.concatenate(([0], boundaries))
        ends = np.concatenate((boundaries, [len(order)]))
        groups = []
        skipped = 0
        for start, end in zip(starts, ends):
            size = int(end - start)
            if size < min_samples:
                skipped += size
                continue
            if size > max_samples:
                raise ValueError(
                    f"date batch size {size} exceeds configured maximum {max_samples}"
                )
            groups.append(np.ascontiguousarray(order[start:end], dtype=np.int64))
        if not groups:
            raise ValueError("no date has enough samples for daily ranking training")

        self.groups = tuple(groups)
        self.shuffle_dates = bool(shuffle_dates)
        self.seed = int(seed)
        self.epoch = 0
        self.skipped_samples = skipped

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        order = np.arange(len(self.groups))
        if self.shuffle_dates:
            rng = np.random.default_rng(self.seed + self.epoch)
            rng.shuffle(order)
        for group_index in order:
            yield self.groups[int(group_index)].tolist()

    def __len__(self):
        return len(self.groups)


class DailyHybridRankLoss(nn.Module):
    """Combine signed IC, sparse pair ordering, and robust standardized regression."""

    def __init__(
        self,
        pair_offsets=PAIR_OFFSETS,
        ic_weight=LOSS_IC_WEIGHT,
        pair_weight=LOSS_PAIR_WEIGHT,
        reg_weight=LOSS_REG_WEIGHT,
        eps=1e-6,
    ):
        super().__init__()
        weights = (ic_weight, pair_weight, reg_weight)
        if any(weight < 0 for weight in weights) or not np.isclose(sum(weights), 1.0):
            raise ValueError("loss weights must be non-negative and sum to one")
        offsets = tuple(int(offset) for offset in pair_offsets)
        if not offsets or any(offset <= 0 for offset in offsets):
            raise ValueError("pair offsets must be positive integers")
        self.pair_offsets = offsets
        self.ic_weight = float(ic_weight)
        self.pair_weight = float(pair_weight)
        self.reg_weight = float(reg_weight)
        self.eps = float(eps)

    def _standardize(self, values):
        centered = values - values.mean()
        scale = torch.sqrt(torch.mean(centered.square()) + self.eps)
        return centered / scale

    def forward(self, prediction, target):
        if prediction.ndim != 1 or target.ndim != 1 or prediction.shape != target.shape:
            raise ValueError("prediction and target must be matching one-dimensional tensors")
        if prediction.numel() < 2:
            raise ValueError("daily rank loss requires at least two samples")

        prediction_z = self._standardize(prediction.float())
        target_z = self._standardize(target.float())
        ic = torch.mean(prediction_z * target_z)
        pair_terms = []
        for offset in self.pair_offsets:
            if offset >= prediction_z.numel():
                continue
            prediction_delta = prediction_z - torch.roll(prediction_z, shifts=offset)
            target_delta = target_z - torch.roll(target_z, shifts=offset)
            direction = torch.sign(target_delta)
            valid = direction != 0
            if torch.any(valid):
                margin = prediction_delta[valid] * direction[valid]
                pair_terms.append(F.softplus(-margin).mean())
        pairwise = (
            torch.stack(pair_terms).mean()
            if pair_terms
            else prediction_z.sum() * 0.0
        )
        regression = F.smooth_l1_loss(prediction_z, target_z)
        loss = (
            self.ic_weight * (1.0 - ic)
            + self.pair_weight * pairwise
            + self.reg_weight * regression
        )
        metrics = {
            "ic": ic.detach(),
            "pairwise": pairwise.detach(),
            "regression": regression.detach(),
        }
        return loss, metrics


class AdaptiveSoftLongShortLoss(nn.Module):
    """Optimize a breadth-controlled differentiable daily long-short utility."""

    def __init__(
        self,
        target_fraction=SOFT_LS_TARGET_FRACTION,
        min_temperature=SOFT_LS_MIN_TEMPERATURE,
        max_temperature=SOFT_LS_MAX_TEMPERATURE,
        bisection_steps=SOFT_LS_BISECTION_STEPS,
        eps=SOFT_LS_EPS,
    ):
        super().__init__()
        if not 0.0 < float(target_fraction) <= 1.0:
            raise ValueError("target_fraction must be in (0, 1]")
        if float(min_temperature) <= 0.0:
            raise ValueError("min_temperature must be positive")
        if float(max_temperature) < float(min_temperature):
            raise ValueError(
                "max_temperature must be at least min_temperature"
            )
        if int(bisection_steps) <= 0:
            raise ValueError("bisection_steps must be positive")
        if float(eps) <= 0.0:
            raise ValueError("eps must be positive")
        self.target_fraction = float(target_fraction)
        self.min_temperature = float(min_temperature)
        self.max_temperature = float(max_temperature)
        self.bisection_steps = int(bisection_steps)
        self.eps = float(eps)

    def _validate_inputs(self, prediction, target):
        if (
            prediction.ndim != 1
            or target.ndim != 1
            or prediction.shape != target.shape
        ):
            raise ValueError(
                "prediction and target must be matching one-dimensional tensors"
            )
        if prediction.numel() < 2:
            raise ValueError("soft long-short loss requires at least two samples")
        if not torch.isfinite(prediction).all():
            raise ValueError("prediction contains non-finite values")
        if not torch.isfinite(target).all():
            raise ValueError("target contains non-finite values")

    def _standardize(self, values):
        centered = values - values.mean()
        scale = torch.sqrt(torch.mean(centered.square()) + self.eps)
        return centered / scale

    @staticmethod
    def _effective_count(weights):
        return weights.square().sum().reciprocal()

    @torch.no_grad()
    def _solve_temperature(self, signed_scores):
        target_count = max(
            2,
            min(
                signed_scores.numel(),
                int(round(signed_scores.numel() * self.target_fraction)),
            ),
        )

        def effective_count(temperature):
            weights = torch.softmax(
                signed_scores / float(temperature), dim=0
            )
            return float(self._effective_count(weights))

        lower = self.min_temperature
        upper = self.max_temperature
        if effective_count(lower) >= target_count:
            return lower
        if effective_count(upper) <= target_count:
            return upper
        for _ in range(self.bisection_steps):
            middle = 0.5 * (lower + upper)
            if effective_count(middle) < target_count:
                lower = middle
            else:
                upper = middle
        return 0.5 * (lower + upper)

    def forward(self, prediction, target):
        self._validate_inputs(prediction, target)
        with torch.autocast(
            device_type=prediction.device.type, enabled=False
        ):
            prediction_z = self._standardize(prediction.float())
            target_z = self._standardize(target.float())
            long_temperature = self._solve_temperature(prediction_z.detach())
            short_temperature = self._solve_temperature(
                -prediction_z.detach()
            )
            long_weight = torch.softmax(
                prediction_z / long_temperature, dim=0
            )
            short_weight = torch.softmax(
                -prediction_z / short_temperature, dim=0
            )
            long_effective = self._effective_count(long_weight)
            short_effective = self._effective_count(short_weight)
            utility = torch.sum(long_weight * target_z) - torch.sum(
                short_weight * target_z
            )
            loss = F.softplus(-utility)
            section_size = prediction_z.new_tensor(
                float(prediction_z.numel())
            )
            metrics = {
                "soft_ls_loss": loss.detach(),
                "soft_ls_utility": utility.detach(),
                "soft_ls_long_temperature": prediction_z.new_tensor(
                    long_temperature
                ),
                "soft_ls_short_temperature": prediction_z.new_tensor(
                    short_temperature
                ),
                "soft_ls_long_effective_count": long_effective.detach(),
                "soft_ls_short_effective_count": short_effective.detach(),
                "soft_ls_long_effective_fraction": (
                    long_effective / section_size
                ).detach(),
                "soft_ls_short_effective_fraction": (
                    short_effective / section_size
                ).detach(),
                "soft_ls_long_max_weight": long_weight.max().detach(),
                "soft_ls_short_max_weight": short_weight.max().detach(),
                "soft_ls_long_weight_sum": long_weight.sum().detach(),
                "soft_ls_short_weight_sum": short_weight.sum().detach(),
                "soft_ls_net_exposure": (
                    long_weight.sum() - short_weight.sum()
                ).detach(),
            }
        return loss, metrics


class V23ADailyLoss(nn.Module):
    """Keep V8B supervision and add one fixed-weight soft portfolio term."""

    def __init__(self, soft_weight=SOFT_LS_WEIGHT):
        super().__init__()
        if float(soft_weight) < 0.0:
            raise ValueError("soft_weight must be non-negative")
        self.soft_weight = float(soft_weight)
        self.base_loss = DailyHybridRankLoss()
        self.soft_loss = AdaptiveSoftLongShortLoss()

    def forward(self, prediction, target):
        base_loss, base_metrics = self.base_loss(prediction, target)
        soft_loss, soft_metrics = self.soft_loss(prediction, target)
        loss = base_loss + self.soft_weight * soft_loss
        metrics = dict(base_metrics)
        metrics["v8b_loss"] = base_loss.detach()
        metrics.update(soft_metrics)
        return loss, metrics

def _make_grad_scaler(enabled):
    amp_module = getattr(torch, "amp", None)
    scaler_type = getattr(amp_module, "GradScaler", None)
    if scaler_type is not None:
        try:
            return scaler_type("cuda", enabled=enabled)
        except TypeError:
            return scaler_type(enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def _autocast_context(enabled):
    amp_module = getattr(torch, "amp", None)
    autocast = getattr(amp_module, "autocast", None)
    if autocast is not None:
        return autocast("cuda", enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)


def build_training_arrays(table, *, query_fn=None, seq_len=SEQ_LEN):
    started = time.perf_counter()
    query = query_fn or _default_query
    pool = query_training_pool(query)
    eligible_index = pd.MultiIndex.from_frame(pool[["date", "instrument"]])
    instruments = sorted(pool["instrument"].unique().tolist())
    capacity = len(pool)
    raw_x = np.empty((capacity, seq_len, len(FEATURE_COLS)), dtype=np.float32)
    raw_y = np.empty(capacity, dtype=np.float32)
    raw_dates = np.empty(capacity, dtype=np.int64)
    offset = 0
    candidate_total = 0
    rejected_total = 0
    allocation_mib = (raw_x.nbytes + raw_y.nbytes + raw_dates.nbytes) / (1024 * 1024)
    LOGGER.info(
        "allocated training storage: capacity=%s allocation_mib=%.2f",
        capacity,
        allocation_mib,
    )

    for chunk_index, instrument_chunk in enumerate(
        iter_chunks(instruments, QUERY_INSTRUMENT_CHUNK), start=1
    ):
        chunk_started = time.perf_counter()
        raw = query_raw_bars(table, instrument_chunk, query_fn=query)
        row_count = len(raw)
        try:
            x_part, y_part, keys = build_raw_sequences(
                raw,
                TRAIN_START,
                TRAIN_END,
                mode="train",
                seq_len=seq_len,
                assume_sorted=True,
            )
        except RuntimeError as exc:
            del raw
            if not str(exc).startswith("no usable sample exists for mode='train'"):
                raise
            LOGGER.info(
                "training chunk skipped: chunk=%s instruments=%s rows=%s candidates=0 retained=0 rejected=0 elapsed_seconds=%.2f",
                chunk_index,
                len(instrument_chunk),
                row_count,
                time.perf_counter() - chunk_started,
            )
            continue
        del raw
        candidate_count = len(y_part)
        candidate_total += candidate_count
        x_part, y_part, _keys, rejected = filter_training_samples(
            x_part,
            y_part,
            keys,
            eligible_index=eligible_index,
        )
        retained_count = len(y_part)
        rejected_total += rejected
        offset = write_training_samples(
            raw_x,
            raw_y,
            raw_dates,
            offset,
            x_part,
            y_part,
            _keys,
        )
        LOGGER.info(
            "training chunk complete: chunk=%s instruments=%s rows=%s candidates=%s retained=%s rejected=%s elapsed_seconds=%.2f",
            chunk_index,
            len(instrument_chunk),
            row_count,
            candidate_count,
            retained_count,
            rejected,
            time.perf_counter() - chunk_started,
        )
        del x_part, y_part, keys, _keys

    if offset == 0:
        raise RuntimeError("no dated CSI 1000 training sample was retained")

    x = raw_x[:offset]
    y = raw_y[:offset]
    date_ids = raw_dates[:offset]
    preprocess_started = time.perf_counter()
    preprocess = fit_preprocess_inplace(x, y)
    preprocess_seconds = time.perf_counter() - preprocess_started
    _assert_finite_array("training tensor", x)
    _assert_finite_array("training label", y)

    tensor_mib = (x.nbytes + y.nbytes + date_ids.nbytes) / (1024 * 1024)
    capacity_utilization = offset / capacity
    rejection_rate = rejected_total / candidate_total if candidate_total else 0.0
    LOGGER.info(
        "built training arrays: samples=%s full_union_size=%s candidate_total=%s retained_total=%s rejected_total=%s allocation_mib=%.2f tensor_mib=%.2f capacity_utilization=%.2f rejection_rate=%.2f preprocess_seconds=%.2f elapsed_seconds=%.2f",
        len(y),
        len(instruments),
        candidate_total,
        offset,
        rejected_total,
        allocation_mib,
        tensor_mib,
        capacity_utilization,
        rejection_rate,
        preprocess_seconds,
        time.perf_counter() - started,
    )
    return x, y, date_ids, preprocess, instruments


def train_model(
    x,
    y,
    date_ids,
    *,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    seed=SEED,
    device=None,
):
    x_array = np.ascontiguousarray(np.asarray(x, dtype=np.float32))
    y_array = np.ascontiguousarray(np.asarray(y, dtype=np.float32))
    dates_array = np.ascontiguousarray(np.asarray(date_ids, dtype=np.int64))
    if x_array.ndim != 3 or x_array.shape[2] != len(FEATURE_COLS):
        raise ValueError("training tensor must have shape (N, seq_len, feature width 10)")
    if y_array.ndim != 1 or y_array.shape[0] != x_array.shape[0]:
        raise ValueError("training labels must be a 1-D array matching training samples")
    if dates_array.ndim != 1 or dates_array.shape[0] != x_array.shape[0]:
        raise ValueError("training date IDs must be a 1-D array matching training samples")
    if x_array.shape[0] == 0:
        raise ValueError("training tensors must contain at least one sample")
    _assert_finite_array("training tensor", x_array)
    _assert_finite_array("training label", y_array)

    seed_everything(seed)
    resolved_device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    model = V2EnhancedTransformer(**MODEL_CONFIG).to(resolved_device)
    parameter_count = validate_parameter_count(model)
    dataset = TensorDataset(torch.from_numpy(x_array), torch.from_numpy(y_array))
    sampler = DateBatchSampler(
        dates_array,
        shuffle_dates=True,
        seed=seed,
        min_samples=MIN_CROSS_SECTION,
        max_samples=batch_size,
    )
    pin_memory = resolved_device.type == "cuda"
    loader = DataLoader(
        dataset,
        batch_sampler=sampler,
        num_workers=0,
        pin_memory=pin_memory,
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    total_steps = max(1, len(loader) * epochs)
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

    def learning_rate_scale(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, learning_rate_scale)
    criterion = V23ADailyLoss()
    amp_enabled = resolved_device.type == "cuda"
    scaler = _make_grad_scaler(amp_enabled)
    history = []
    LOGGER.info(
        "training start: device=%s parameters=%s samples=%s dates=%s epochs=%s "
        "max_date_batch=%s skipped_small_date_samples=%s",
        resolved_device,
        parameter_count,
        len(dataset),
        len(sampler),
        epochs,
        batch_size,
        sampler.skipped_samples,
    )
    for epoch in range(epochs):
        epoch_started = time.perf_counter()
        sampler.set_epoch(epoch)
        model.train()
        total_loss = 0.0
        total_ic = 0.0
        total_pairwise = 0.0
        total_regression = 0.0
        total_soft_ls_loss = 0.0
        total_soft_ls_utility = 0.0
        total_soft_ls_long_temperature = 0.0
        total_soft_ls_short_temperature = 0.0
        total_soft_ls_long_effective_fraction = 0.0
        total_soft_ls_short_effective_fraction = 0.0
        total_soft_ls_long_max_weight = 0.0
        total_soft_ls_short_max_weight = 0.0
        total_dates = 0
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(resolved_device, non_blocking=pin_memory)
            batch_y = batch_y.to(resolved_device, non_blocking=pin_memory)
            optimizer.zero_grad(set_to_none=True)
            with _autocast_context(amp_enabled):
                prediction = model(batch_x)
                if prediction.shape != batch_y.shape:
                    raise ValueError(
                        f"model output shape {tuple(prediction.shape)} does not match labels {tuple(batch_y.shape)}"
                    )
                loss, metrics = criterion(prediction, batch_y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at epoch {epoch + 1}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += float(loss.detach())
            total_ic += float(metrics["ic"])
            total_pairwise += float(metrics["pairwise"])
            total_regression += float(metrics["regression"])
            total_soft_ls_loss += float(metrics["soft_ls_loss"])
            total_soft_ls_utility += float(metrics["soft_ls_utility"])
            total_soft_ls_long_temperature += float(
                metrics["soft_ls_long_temperature"]
            )
            total_soft_ls_short_temperature += float(
                metrics["soft_ls_short_temperature"]
            )
            total_soft_ls_long_effective_fraction += float(
                metrics["soft_ls_long_effective_fraction"]
            )
            total_soft_ls_short_effective_fraction += float(
                metrics["soft_ls_short_effective_fraction"]
            )
            total_soft_ls_long_max_weight += float(
                metrics["soft_ls_long_max_weight"]
            )
            total_soft_ls_short_max_weight += float(
                metrics["soft_ls_short_max_weight"]
            )
            total_dates += 1
        epoch_metrics = {
            "loss": total_loss / total_dates,
            "ic": total_ic / total_dates,
            "pairwise": total_pairwise / total_dates,
            "regression": total_regression / total_dates,
            "soft_ls_loss": total_soft_ls_loss / total_dates,
            "soft_ls_utility": total_soft_ls_utility / total_dates,
            "soft_ls_long_temperature": (
                total_soft_ls_long_temperature / total_dates
            ),
            "soft_ls_short_temperature": (
                total_soft_ls_short_temperature / total_dates
            ),
            "soft_ls_long_effective_fraction": (
                total_soft_ls_long_effective_fraction / total_dates
            ),
            "soft_ls_short_effective_fraction": (
                total_soft_ls_short_effective_fraction / total_dates
            ),
            "soft_ls_long_max_weight": (
                total_soft_ls_long_max_weight / total_dates
            ),
            "soft_ls_short_max_weight": (
                total_soft_ls_short_max_weight / total_dates
            ),
        }
        history.append(epoch_metrics)
        LOGGER.info(
            "training epoch complete: epoch=%s loss=%.8f signed_ic=%.6f "
            "pairwise=%.6f regression=%.6f soft_ls=%.6f utility=%.6f "
            "effective_long=%.4f effective_short=%.4f "
            "learning_rate=%.8g elapsed_seconds=%.2f",
            epoch + 1,
            epoch_metrics["loss"],
            epoch_metrics["ic"],
            epoch_metrics["pairwise"],
            epoch_metrics["regression"],
            epoch_metrics["soft_ls_loss"],
            epoch_metrics["soft_ls_utility"],
            epoch_metrics["soft_ls_long_effective_fraction"],
            epoch_metrics["soft_ls_short_effective_fraction"],
            optimizer.param_groups[0]["lr"],
            time.perf_counter() - epoch_started,
        )
    model = model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return model, history


def _normalize_requested_dates(start_date, end_date):
    start_day = pd.Timestamp(start_date).normalize()
    end_day = pd.Timestamp(end_date).normalize()
    if pd.isna(start_day) or pd.isna(end_day):
        raise ValueError("start_date and end_date must be valid dates")
    if start_day > end_day:
        raise ValueError("start_date must be on or before end_date")
    end_timestamp = end_day + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    return start_day, end_day, end_timestamp


def _date_filter_value(timestamp):
    if timestamp == timestamp.normalize():
        return timestamp.strftime("%Y-%m-%d")
    return timestamp.strftime("%Y-%m-%d %H:%M:%S")


def _pool_table(datasources):
    if datasources is None:
        return POOL_TABLE
    for key in ("instruments", "instrument_pool", "pool", "e2e_instruments"):
        if key in datasources:
            return datasources[key]
    return POOL_TABLE


def _normalise_pool_frame(pool, start_day, end_day):
    required = {"date", "instrument"}
    missing = sorted(required.difference(pool.columns))
    if missing:
        raise ValueError(f"official pool is missing required columns: {missing}")

    normalised = pool.loc[:, ["date", "instrument"]].copy()
    normalised["date"] = pd.to_datetime(normalised["date"], errors="coerce")
    if normalised["date"].isna().any():
        raise ValueError("official pool contains null or invalid dates")
    normalised["date"] = normalised["date"].dt.normalize()
    if normalised["instrument"].isna().any():
        raise ValueError("official pool contains null instruments")
    normalised["instrument"] = normalised["instrument"].astype(str)
    normalised = normalised.loc[
        (normalised["date"] >= start_day) & (normalised["date"] <= end_day)
    ].reset_index(drop=True)
    if normalised.empty:
        raise ValueError("official pool is empty for the requested interval")
    if normalised.duplicated(["date", "instrument"]).any():
        raise ValueError("official pool contains duplicate date/instrument keys")
    return normalised.sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)


def query_official_pool(datasources, start_day, end_day, query_fn=None):
    started = time.perf_counter()
    query = query_fn or _default_query
    table = _pool_table(datasources)
    sql = f"select date, instrument from {table} order by date, instrument"
    start_filter = _date_filter_value(start_day)
    end_filter = _date_filter_value(end_day)
    LOGGER.info(
        "inference pool query: table=%s date_start=%s date_end=%s",
        table,
        start_filter,
        end_filter,
    )
    frame = _query_result_to_frame(
        query(
            sql,
            filters={"date": [start_filter, end_filter]},
            compression=False,
        )
    )
    normalised = _normalise_pool_frame(frame, start_day, end_day)
    LOGGER.info(
        "inference pool query complete: table=%s rows=%s elapsed_seconds=%.2f",
        table,
        len(normalised),
        time.perf_counter() - started,
    )
    return normalised


def query_raw_bars_for_interval(
    table,
    instruments,
    start_timestamp,
    end_timestamp,
    query_fn=None,
):
    started = time.perf_counter()
    instrument_list = [str(instrument) for instrument in instruments]
    if len(instrument_list) > QUERY_INSTRUMENT_CHUNK:
        raise ValueError(
            f"bar query cannot request more than {QUERY_INSTRUMENT_CHUNK} instruments, got {len(instrument_list)}"
        )

    columns = ("date", "instrument", *FEATURE_COLS)
    sql = f"select {', '.join(columns)} from {table} order by instrument, date"
    query = query_fn or _default_query
    start_filter = _date_filter_value(start_timestamp)
    end_filter = _date_filter_value(end_timestamp)
    LOGGER.info(
        "inference raw-bar query: table=%s date_start=%s date_end=%s chunk_size=%s",
        table,
        start_filter,
        end_filter,
        len(instrument_list),
    )
    frame = _query_result_to_frame(
        query(
            sql,
            filters={
                "date": [start_filter, end_filter],
                "instrument": instrument_list,
            },
            compression=True,
        )
    )
    LOGGER.info(
        "inference raw-bar query complete: table=%s instruments=%s rows=%s elapsed_seconds=%.2f",
        table,
        len(instrument_list),
        len(frame),
        time.perf_counter() - started,
    )
    return frame


def predict_sequences(model, x, batch_size, device):
    """Run finite 1-D transformer inference under torch.inference_mode."""
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")
    x_array = np.ascontiguousarray(np.asarray(x, dtype=np.float32))
    if x_array.ndim != 3:
        raise ValueError("inference tensor must have shape (N, seq_len, feature width)")
    if x_array.shape[0] == 0:
        return np.empty(0, dtype=np.float32)
    _assert_finite_array("inference tensor", x_array)

    resolved_device = torch.device(device)
    model = model.to(resolved_device)
    model.eval()
    batches = []
    with torch.inference_mode():
        for start in range(0, x_array.shape[0], batch_size):
            batch = torch.from_numpy(x_array[start : start + batch_size]).to(resolved_device)
            prediction = model(batch)
            if prediction.ndim != 1 or prediction.shape[0] != batch.shape[0]:
                raise ValueError("model prediction must be a 1-D tensor matching the batch size")
            batches.append(prediction.detach().cpu().numpy())

    scores = np.concatenate(batches).astype(np.float32, copy=False)
    if scores.shape != (x_array.shape[0],):
        raise ValueError("model prediction must be one-dimensional")
    _assert_finite_array("model predictions", scores)
    return np.ascontiguousarray(scores)


def check_prediction_coverage(joined):
    """Return daily missing rates and reject any day with more than 40% missing predictions."""
    required = {"date", "instrument", "score"}
    missing_columns = sorted(required.difference(joined.columns))
    if missing_columns:
        raise ValueError(f"joined predictions are missing required columns: {missing_columns}")

    frame = joined.loc[:, ["date", "instrument", "score"]].copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
    missing_rates = frame["score"].isna().groupby(frame["date"]).mean().sort_index()
    bad_days = missing_rates[missing_rates > 0.40]
    if not bad_days.empty:
        formatted = ", ".join(f"{day.date()}={rate:.1%}" for day, rate in bad_days.items())
        raise RuntimeError(f"pre-fill model missing rate exceeds 40%: {formatted}")
    return missing_rates


def fill_missing_scores(result):
    """Fill allowed missing scores by daily median, then zero for fully missing days."""
    frame = result.copy()
    if "score" not in frame:
        raise ValueError("result must contain a score column")
    frame["score"] = pd.to_numeric(frame["score"], errors="coerce")
    daily_median = frame.groupby(pd.to_datetime(frame["date"]).dt.normalize())["score"].transform("median")
    frame["score"] = frame["score"].fillna(daily_median).fillna(0.0).astype(float)
    return frame


def _validated_keys(frame, name):
    if frame["date"].isna().any():
        raise ValueError(f"{name} contains null dates")
    dates = pd.to_datetime(frame["date"], errors="coerce")
    if dates.isna().any():
        raise ValueError(f"{name} contains invalid dates")
    if not (dates == dates.dt.normalize()).all():
        raise ValueError(f"{name} dates must be normalized")

    if frame["instrument"].isna().any():
        raise ValueError(f"{name} contains null instruments")
    if not frame["instrument"].map(lambda value: isinstance(value, str)).all():
        raise ValueError(f"{name} instruments must be strings")

    keys = pd.DataFrame({"date": dates.dt.normalize(), "instrument": frame["instrument"]})
    if keys.duplicated(["date", "instrument"]).any():
        raise RuntimeError(f"{name} contains duplicate date/instrument keys")
    return keys


def validate_prediction_frame(frame, start_date, end_date):
    """Validate finite unique scores inside the requested date interval."""
    expected_columns = ["date", "instrument", "score"]
    if list(frame.columns) != expected_columns:
        raise ValueError(f"prediction columns must be exactly {expected_columns}")
    checked = frame.copy()
    keys = _validated_keys(checked, "prediction frame")
    start_day, end_day, _ = _normalize_requested_dates(start_date, end_date)
    if (keys["date"] < start_day).any() or (keys["date"] > end_day).any():
        raise ValueError("prediction frame contains dates outside the requested interval")
    scores = pd.to_numeric(checked["score"], errors="coerce")
    if scores.isna().any():
        raise ValueError("prediction scores must be finite numeric values")
    _assert_finite_array("prediction scores", scores.to_numpy(dtype=float))
    return pd.DataFrame(
        {
            "date": keys["date"],
            "instrument": keys["instrument"],
            "score": scores.astype(float),
        }
    ).sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)


def validate_output(result, expected_pool):
    """Validate the exact public submission contract and return sorted output columns."""
    expected_columns = ["date", "instrument", "score"]
    if list(result.columns) != expected_columns:
        raise ValueError(f"result columns must be exactly {expected_columns}")

    result_frame = result.copy()
    expected_frame = expected_pool.loc[:, ["date", "instrument"]].copy()
    result_keys = _validated_keys(result_frame, "result")
    expected_keys = _validated_keys(expected_frame, "expected pool")

    scores = pd.to_numeric(result_frame["score"], errors="coerce")
    score_values = scores.to_numpy(dtype=float)
    if scores.isna().any():
        raise ValueError("result scores must be finite numeric values")
    _assert_finite_array("result scores", score_values)

    result_key_index = pd.MultiIndex.from_frame(result_keys)
    expected_key_index = pd.MultiIndex.from_frame(expected_keys)
    if set(result_key_index) != set(expected_key_index):
        raise RuntimeError("result date/instrument keys must exactly match the official pool")

    if scores.nunique(dropna=False) < 2:
        raise ValueError("result scores are constant overall")

    checked = pd.DataFrame(
        {
            "date": result_keys["date"],
            "instrument": result_keys["instrument"],
            "score": scores.astype(float),
        }
    )
    return checked.sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)


def run_optional_evaluation(label, score_fn, evaluator):
    """Run an optional scoring/evaluation branch without aborting required work."""
    empty = pd.DataFrame(columns=["date", "instrument", "score"])
    try:
        scores = score_fn()
        if not isinstance(scores, pd.DataFrame):
            raise TypeError("optional score function must return a DataFrame")
        if scores.empty:
            LOGGER.warning("optional %s evaluation skipped: no score data", label)
            return empty, None
        return scores, evaluator(factor_data=scores, show=True)
    except Exception:
        LOGGER.warning(
            "optional %s evaluation skipped because data is unavailable",
            label,
            exc_info=True,
        )
        return empty, None


def train_fixed_period(query_fn=None):
    """Train V40A once on the fixed 2019-2024 period."""
    started = time.perf_counter()
    seed_everything(SEED)
    query = query_fn or _default_query

    train_x, train_y, train_dates, preprocess, train_instruments = build_training_arrays(
        TRAIN_TABLE,
        query_fn=query,
        seq_len=SEQ_LEN,
    )
    training_samples = len(train_y)
    model, history = train_model(
        train_x,
        train_y,
        train_dates,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        seed=SEED,
    )
    del train_x, train_y, train_dates
    gc.collect()
    LOGGER.info(
        "fixed-period training complete: samples=%s instruments=%s epochs=%s elapsed_seconds=%.2f",
        training_samples,
        len(train_instruments),
        len(history),
        time.perf_counter() - started,
    )
    return model, preprocess


def score_interval(
    model,
    preprocess,
    datasources,
    start_date,
    end_date,
    query_fn=None,
):
    """Score one requested interval using an already trained V40A model."""
    started = time.perf_counter()
    start_day, end_day, end_timestamp = _normalize_requested_dates(
        start_date, end_date
    )
    infer_table = pick_bar1m_table(datasources)
    query = query_fn or _default_query

    expected_pool = query_official_pool(
        datasources,
        start_day,
        end_day,
        query_fn=query,
    )
    query_start = start_day - pd.Timedelta(days=INFER_BUFFER_DAYS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    predictions = []
    instruments = sorted(expected_pool["instrument"].unique().tolist())
    for chunk_index, instrument_chunk in enumerate(
        iter_chunks(instruments, QUERY_INSTRUMENT_CHUNK), start=1
    ):
        chunk_started = time.perf_counter()
        raw = query_raw_bars_for_interval(
            infer_table,
            instrument_chunk,
            query_start,
            end_timestamp,
            query_fn=query,
        )
        raw_rows = len(raw)
        try:
            raw_x, _, keys = build_raw_sequences(
                raw,
                query_start,
                end_timestamp,
                mode="infer",
                seq_len=SEQ_LEN,
                assume_sorted=True,
            )
        except RuntimeError as exc:
            del raw
            if "no usable sample exists" in str(exc):
                LOGGER.info(
                    "inference chunk skipped: chunk=%s instruments=%s rows=%s elapsed_seconds=%.2f",
                    chunk_index,
                    len(instrument_chunk),
                    raw_rows,
                    time.perf_counter() - chunk_started,
                )
                continue
            raise
        del raw

        keys = keys.copy()
        keys["date"] = pd.to_datetime(keys["date"]).dt.normalize()
        keys["instrument"] = keys["instrument"].astype(str)
        requested = (keys["date"] >= start_day) & (keys["date"] <= end_day)
        if not requested.any():
            continue
        x = apply_preprocess(raw_x[requested.to_numpy()], preprocess)
        scores = predict_sequences(model, x, INFER_BATCH_SIZE, device)
        chunk = keys.loc[
            requested, ["date", "instrument"]
        ].reset_index(drop=True)
        chunk["score"] = scores
        predictions.append(chunk)
        LOGGER.info(
            "inference chunk complete: chunk=%s instruments=%s rows=%s predictions=%s elapsed_seconds=%.2f",
            chunk_index,
            len(instrument_chunk),
            raw_rows,
            len(chunk),
            time.perf_counter() - chunk_started,
        )

    prediction_frame = (
        pd.concat(predictions, ignore_index=True)
        if predictions
        else pd.DataFrame(columns=["date", "instrument", "score"])
    )
    if prediction_frame.duplicated(["date", "instrument"]).any():
        raise RuntimeError("duplicate prediction keys are not allowed")
    joined = expected_pool.merge(
        prediction_frame,
        on=["date", "instrument"],
        how="left",
        validate="one_to_one",
    )
    missing_rates = check_prediction_coverage(joined)
    filled = fill_missing_scores(joined)[["date", "instrument", "score"]]
    filled = validate_prediction_frame(filled, start_day, end_day)
    result = validate_output(filled, expected_pool)
    LOGGER.info(
        "interval scoring complete: rows=%s days=%s max_missing_rate=%.4f elapsed_seconds=%.2f",
        len(result),
        result["date"].nunique(),
        float(missing_rates.max()),
        time.perf_counter() - started,
    )
    return result


def main(datasources, start_date, end_date):
    """Train on the fixed 2019-2024 set, then score the injected interval."""
    started = time.perf_counter()
    model, preprocess = train_fixed_period()
    result = score_interval(
        model,
        preprocess,
        datasources,
        start_date,
        end_date,
    )
    LOGGER.info("realtime run complete: elapsed_seconds=%.2f", time.perf_counter() - started)
    return result


if __name__ == "__main__":
    from bigmodule import M

    # 只用 1 分钟 K 线作为输入数据
    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
    }

    # 固定训练一次，再分别查询和评估 2024、2025。
    model, preprocess = train_fixed_period()

    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    LOGGER.warning(LOCAL_SMOKE_WARNING)
    LOGGER.info("计算分数: start=%s end=%s", start_date, end_date)
    score_data_2024 = score_interval(
        model,
        preprocess,
        datasources,
        start_date,
        end_date,
    )
    if score_data_2024.empty:
        raise RuntimeError("2024 evaluation data is empty")
    print(score_data_2024.head())

    LOGGER.info("开始评估分数（仅流程冒烟，不是样本外成绩）")
    result_2024 = M.bigalpha_eval._latest(
        factor_data=score_data_2024,
        show=True,
    )

    score_data_2025, result_2025 = run_optional_evaluation(
        "2025",
        lambda: score_interval(
            model,
            preprocess,
            datasources,
            "2025-01-01 00:00:00",
            "2025-12-31 23:59:59",
        ),
        M.bigalpha_eval._latest,
    )
    if score_data_2025.empty:
        LOGGER.warning("未取得可评估的 2025 行情，已安全跳过 2025 样本外评估")
    else:
        print(score_data_2025.head())
